# Flux LoRA Train

⚠️ **Remember to copy this notebook in your Drive and rename.**


**This workflow uses AI Toolkit'**

[jhj0517/finetuning-notebooks](https://github.com/jhj0517/finetuning-notebooks)

[ostris/ai-toolkit](https://github.com/ostris/ai-toolkit)

**HuggingFace Alternative**

[README_flux.md](https://github.com/huggingface/diffusers/blob/main/examples/dreambooth/README_flux.md)

[train_dreambooth_lora_flux.py](https://github.com/huggingface/diffusers/blob/main/examples/dreambooth/train_dreambooth_lora_flux.py)

[test_dreambooth_lora_flux.py](https://github.com/huggingface/diffusers/blob/main/examples/dreambooth/test_dreambooth_lora_flux.py)

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

##Confirm using A100 GPU

You absolutely need an A100 to train a Flux LoRA. Sometimes Colab says that it is connected to an A100 while secretly connecting to a lower GPU. Confirm below that you actually connected to an A100 as this isn't going to work if you are connected to a lesser GPU.

In [ ]:
!nvidia-smi

## Install

In [ ]:
!git clone -b fix/numpy-error https://github.com/jhj0517/ai-toolkit.git
%cd ai-toolkit
!git submodule update --init --recursive > /dev/null 2>&1
!pip install --quiet -r requirements.txt

#Downgrade Numpy
!pip uninstall -y numpy
!pip install --quiet --force-reinstall --no-deps numpy==1.26.3

In [ ]:
# Restart session
import os
os.kill(os.getpid(), 9)
print("IGNORE: 'Your session crashed for an unknown reason'. This was done intentionally.")
# Continue to the next cell and keep going after runtime restarts

## Setup

In [ ]:
# Fix numpy incompatibility from https://github.com/ostris/ai-toolkit/issues/267
import numpy
print("Numpy should be 1.26.3. The version currently being used is: " + numpy.__version__)

## Mount Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

## Hugging Face Token

In [ ]:
# Sign up at Hugging Face and create a "Read" access token (not the default "Fine-Grained" token).
# Click the 🔑 "Secrets" icon in the left sidebar.
# Enable Notebook Access, Set the Name to "HF_TOKEN", Paste your token as the Value

from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

## Set Directories

In [ ]:
DATASET_DIR = '/content/drive/MyDrive/iaac_genai/workflows/03_FINETUNING/02_FLUX.1/dataset_FLUX.1'
OUTPUT_DIR = '/content/drive/MyDrive/iaac_genai/workflows/03_FINETUNING/02_FLUX.1/weights_FLUX.1'
LORA_NAME = 'bof_aar_lacha'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Login to Weights & Biases with token

In [ ]:
# Get your API token from https://wandb.ai/settings
# Click the empty space after "quit:" to enter code
!pip install wandb -q
import wandb
wandb.login()

## Config

In [ ]:
import sys
sys.path.append('/content/ai-toolkit')
from toolkit.job import run_job
from collections import OrderedDict
from PIL import Image
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
# Model
repo_id_or_path = 'black-forest-labs/FLUX.1-dev'
quantize = False
dtype = "bf16" #"float16"

# Training Steps (max_steps)
# More steps = more learning, higher risk of overfitting
# Fewer steps = less learning, may underfit
# 1000  = quick test / smoke run
# 1500  = good for small datasets with low LR
# 2000  = default starting point (30 images)
# 3000  = larger datasets or lower LR runs
# 4000  = high variety datasets, risk of overfit on small sets
steps = 2000

# Saving checkpoints
# Saves a checkpoint every save_every steps
# Deletes oldest checkpoint once max_step_saves_to_keep is exceeded
save_every = 250
max_step_saves_to_keep = 10

# Sampling
sample_every = 250
sample_seed = 77
sample_steps = 50
width = 1024
height = 1024

# Learning Rate
# Higher = faster learning, more risk of overfitting
# Lower  = slower learning, more stable convergence
# 4e-4 = 0.0004  (FLUX.1 default, aggressive)
# 2e-4 = 0.0002  (moderate)
# 1e-4 = 0.0001  (conservative, recommended for FLUX.2)
# 5e-5 = 0.00005 (slow, stable, good for small datasets)
# 1e-5 = 0.00001 (very slow, use if overfitting persists)
lr = 4e-4

# Replace Prompts with your own first three or favorite prompts
sample_prompt_1 = "bof_aar_lacha postmodern Mediterranean apartment complex, swimming pool in foreground, concrete, glass, metal, layered volumes, Coral red, dusty rose pink, cobalt blue,, Mediterranean coastal, Elevated, wide shot."
sample_prompt_2 = "bof_aar_lacha postmodern Mediterranean apartment complex, swimming pool in foreground, concrete, glass, metal, clean lines, pink, purple, black, white,, Poolside, leisure, vacation, luxury, , High angle, looking down."
sample_prompt_3 = "bof_aar_lacha postmodern Mediterranean apartment complex, swimming pool in foreground, castle-like structure, geometric form, concrete, glass, metal, dusty rose pink, coral red, cobalt blu, coastal, luxurious, vacation, aerial, high."
sample_prompts = [sample_prompt_1, sample_prompt_2, sample_prompt_3]


# Advanced
# Lora Network
linear = 16
linear_alpha = 16

# Dataset
caption_ext = "txt"
caption_dropout_rate = 0.05
shuffle_tokens = False
cache_latents_to_disk = True
resolution = [1024]

# Training
performance_log_every = 1000
train_only_specific_layers = False
only_if_contains = ["transformer.single_transformer_blocks.7.proj_out", "transformer.single_transformer_blocks.20.proj_out"]
batch_size = 1
gradient_accumulation_steps = 1
train_dtype = "bf16"
train_unet = True
train_text_encoder = True
content_or_style = 'balanced'
gradient_checkpointing = True
noise_scheduler = 'flowmatch'
optimizer = 'adamw8bit'
use_ema = True
ema_decay = 0.99

## Train

In [ ]:
job_to_run = OrderedDict([
    ('job', 'extension'),
    ('config', OrderedDict([
        ('name', LORA_NAME),
        ('process', [
            OrderedDict([
                ('type', 'sd_trainer'),
                ('training_folder', OUTPUT_DIR),
                ('performance_log_every', 1000),
                ('device', 'cuda:0'),
                ('network', OrderedDict([
                    ('type', 'lora'),
                    ('linear', linear),
                    ('linear_alpha', linear_alpha),
                    ('network_kwargs', OrderedDict([
                      ("prefix", [
                          "lora_te1.text_model.encoder",
                          "transformer.single_transformer_blocks",
                          "transformer.transformer_blocks"
                      ])
                    ]))
                ])),
                ('save', OrderedDict([
                    ('dtype', dtype),
                    ('save_every', save_every),
                    ('max_step_saves_to_keep', max_step_saves_to_keep)
                ])),
                ('datasets', [
                    OrderedDict([
                        ('folder_path', DATASET_DIR),
                        ('caption_ext', caption_ext),
                        ('caption_dropout_rate', caption_dropout_rate),
                        ('shuffle_tokens', shuffle_tokens),
                        ('cache_latents_to_disk', cache_latents_to_disk),
                        ('resolution', resolution)
                    ])
                ]),
                ('train', OrderedDict([
                    ('batch_size', batch_size),
                    ('steps', steps),
                    ('gradient_accumulation_steps', gradient_accumulation_steps),
                    ('train_unet', train_unet),
                    ('train_text_encoder', train_text_encoder),
                    ('content_or_style', content_or_style),
                    ('gradient_checkpointing', gradient_checkpointing),
                    ('noise_scheduler', noise_scheduler),
                    ('optimizer', optimizer),
                    ('lr', lr),
                    ('ema_config', OrderedDict([
                        ('use_ema', use_ema),
                        ('ema_decay', ema_decay)
                    ])),
                    ('dtype', train_dtype)
                ])),
                ('model', OrderedDict([
                    ('name_or_path', repo_id_or_path),
                    ('is_flux', True),
                    ('quantize', quantize),
                ])),
                ('sample', OrderedDict([
                    ('sampler', 'flowmatch'),
                    ('sample_every', sample_every),
                    ('width', width),
                    ('height', height),
                    ('prompts', sample_prompts),
                    ('neg', ''),
                    ('seed', sample_seed),
                    ('walk_seed', True),
                    ('guidance_scale', 4),
                    ('sample_steps', sample_steps)
                ]))
            ])
        ])
    ])),
    ('meta', OrderedDict([
        ('name', '[name]'),
        ('version', '1.0')
    ]))
])

# Conditional Parameters
if train_only_specific_layers:
    network = job_to_run["config"]["process"][0]["network"]
    network_kwargs = network.setdefault("network_kwargs", OrderedDict())
    network_kwargs["only_if_contains"] = only_if_contains


wandb.init(project='flux-lora', name=LORA_NAME, resume='allow')
run_job(job_to_run)

## Disconnect

In [ ]:
from google.colab import runtime
runtime.unassign()